# Mamba vs. Transformer Porównanie zdolności klasyfikacji na zbiorze danych IMDb

## 1. Setup i biblioteki

Możemy to pominąć i wykonać komendę `uv sync` jeśli notebook uruchamiamy lokalnie

### 1.1 Instalacja bibliotek

In [ ]:
# Example for CUDA 12.3+ and PyTorch 2.x (Adjust URL based on your exact versions if needed)
!pip install https://github.com/state-spaces/mamba/releases/download/v2.2.2/mamba_ssm-2.2.2+cu122torch2.3cxx11abiFALSE-cp312-cp312-linux_x86_64.whl

In [ ]:
!pip uninstall -y torch torchvision torchaudio # Uninstall any existing versions to avoid conflicts
!pip install torch==2.3.0 torchvision==0.18.0 torchaudio==2.3.0 --index-url https://download.pytorch.org/whl/cu121


In [ ]:
!pip install transformers datasets accelerate evaluate

### 1.2 Importy

In [6]:
import torch
import datasets
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer, DataCollatorWithPadding
import evaluate
import numpy as np

In [7]:
import evaluate
import numpy as np

# Helper function to compute metrics
def compute_metrics(eval_pred):
    load_accuracy = evaluate.load("accuracy")
    load_f1 = evaluate.load("f1")
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    accuracy = load_accuracy.compute(predictions=predictions, references=labels)["accuracy"]
    f1 = load_f1.compute(predictions=predictions, references=labels, average="weighted")["f1"]
    return {"accuracy": accuracy, "f1": f1}

## 2. Wczytanie datasetu i preprocessing

We will load the `imdb` dataset from the Hugging Face `datasets` library, which contains movie reviews labeled as positive or negative sentiment. After loading, we will tokenize the text and prepare the data for model training.

In [9]:
print("Is CUDA available?", torch.cuda.is_available())

Is CUDA available? True


In [10]:
dataset = datasets.load_dataset('stanfordnlp/imdb')

print("Dataset loaded successfully:")
print(dataset)

Dataset loaded successfully:
DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 25000
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 25000
    })
    unsupervised: Dataset({
        features: ['text', 'label'],
        num_rows: 50000
    })
})


In [11]:
print("\nExample from training set:")
print(dataset['train'][0])


Example from training set:
{'text': 'I rented I AM CURIOUS-YELLOW from my video store because of all the controversy that surrounded it when it was first released in 1967. I also heard that at first it was seized by U.S. customs if it ever tried to enter this country, therefore being a fan of films considered "controversial" I really had to see this for myself.<br /><br />The plot is centered around a young Swedish drama student named Lena who wants to learn everything she can about life. In particular she wants to focus her attentions to making some sort of documentary on what the average Swede thought about certain political issues such as the Vietnam War and race issues in the United States. In between asking politicians and ordinary denizens of Stockholm about their opinions on politics, she has sex with her drama teacher, classmates, and married men.<br /><br />What kills me about I AM CURIOUS-YELLOW is that 40 years ago, this was considered pornographic. Really, the sex and nudi

### 2.2 Tokenizacja

We need to convert the text reviews into numerical tokens that the models can understand. We'll use a `AutoTokenizer` suitable for our chosen Transformer baseline (e.g., DistilBERT) and apply it to the entire dataset. For Mamba, a similar tokenization strategy will be used.

## 3. Podejście 1: Transformer (Baseline)

In this section, we will implement and train a Transformer model. We'll start with fine-tuning a pre-trained model like DistilBERT, which is a good balance between performance and computational cost. We'll define the model, training arguments, and use the `Trainer` API for training.

### Fine-tuning DistilBERT

We will fine-tune a `distilbert-base-uncased` model for sequence classification. This involves loading the pre-trained model, configuring training arguments, and using the `Trainer` API to manage the training and evaluation process.

In [4]:
import time

from transformers import TrainerCallback

def fineTuneBert(max_length=512,batch_size=32):
    tokenizer = AutoTokenizer.from_pretrained("distilbert/distilbert-base-uncased")
    def tokenize_function(examples):
        return tokenizer(examples["text"], truncation=True, max_length=max_length) # Default max_length, will be varied in scaling analysis

    # Apply tokenization to the dataset
    tokenized_dataset = dataset.map(tokenize_function, batched=True)


    # Prepare data for training
    tokenized_dataset = tokenized_dataset.remove_columns(["text"])
    tokenized_dataset = tokenized_dataset.rename_column("label", "labels")
    tokenized_dataset.set_format("torch")

    # Create data collator
    data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

    class TimingCallback(TrainerCallback):
        def on_epoch_begin(self, args, state, control, **kwargs):
            self.epoch_start_time = time.time()

        def on_epoch_end(self, args, state, control, **kwargs):
            epoch_end_time = time.time()
            epoch_duration = epoch_end_time - self.epoch_start_time
            print(f"Epoch {state.epoch:.0f} completed in {epoch_duration:.2f} seconds")

    model = AutoModelForSequenceClassification.from_pretrained("distilbert/distilbert-base-uncased", num_labels=2)

    # Define training arguments
    training_args = TrainingArguments(
        output_dir="./results",
        eval_strategy="epoch",
        learning_rate=2e-5,
        per_device_train_batch_size=batch_size,
        per_device_eval_batch_size=batch_size,
        num_train_epochs=5,
        weight_decay=0.01,
        logging_dir='./logs',
        logging_steps=1, # Changed from 10 to 1 for more frequent updates
        report_to="tensorboard" # Add this for loss tracking
    )

    # Initialize Trainer
    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=tokenized_dataset["train"],
        eval_dataset=tokenized_dataset["test"],
        data_collator=data_collator,
        compute_metrics=compute_metrics,
        callbacks=[TimingCallback()] # Add TensorBoardCallback and TimingCallback
    )

    results = trainer.evaluate()
    print("\nDistilBERT Initial Evaluation Results:")
    print(results)

    # Train the model
    trainer.train()

    # Evaluate the model
    results = trainer.evaluate()
    print("\nDistilBERT Evaluation Results:")
    print(results)

In [11]:
fineTuneBert(128)

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert/distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
pre_classifier.weight   | MISSING    | 
pre_classifier.bias     | MISSING    | 
classifier.weight       | MISSING    | 
classifier.bias         | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
[transformers] `logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


Training Loss,Validation Loss,Epoch,Accuracy,F1
No log,0.699334,0,0.376000,0.335861



DistilBERT Initial Evaluation Results:
{'eval_loss': 0.6993342041969299, 'eval_accuracy': 0.376, 'eval_f1': 0.3358612429812539}


Epoch,Training Loss,Validation Loss,Accuracy,F1
1,0.110386,0.300068,0.868760,0.868748
2,0.174816,0.299552,0.875600,0.875592
3,0.042987,0.333531,0.876120,0.875995
4,0.063415,0.378347,0.874800,0.874766
5,0.009894,0.416754,0.875120,0.875117


...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Epoch 5 completed in 68.32 seconds


Training Loss,Validation Loss,Epoch,Accuracy,F1
0.009894,0.416754,5,0.875120,0.875117



DistilBERT Evaluation Results:
{'eval_loss': 0.41675418615341187, 'eval_accuracy': 0.87512, 'eval_f1': 0.875117122698507}


In [12]:
fineTuneBert(256)

Map:   0%|          | 0/25000 [00:00<?, ? examples/s]

Map:   0%|          | 0/25000 [00:00<?, ? examples/s]

Map:   0%|          | 0/50000 [00:00<?, ? examples/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert/distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
pre_classifier.weight   | MISSING    | 
pre_classifier.bias     | MISSING    | 
classifier.weight       | MISSING    | 
classifier.bias         | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
[transformers] `logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Epoch 5 completed in 138.61 seconds


Training Loss,Validation Loss,Epoch,Accuracy,F1
0.009905,0.321185,5,0.910040,0.910033



DistilBERT Evaluation Results:
{'eval_loss': 0.32118549942970276, 'eval_accuracy': 0.91004, 'eval_f1': 0.9100332216870016}


In [12]:
fineTuneBert(512,batch_size=16)

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert/distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
pre_classifier.weight   | MISSING    | 
pre_classifier.bias     | MISSING    | 
classifier.weight       | MISSING    | 
classifier.bias         | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
[transformers] `logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


Training Loss,Validation Loss,Epoch,Accuracy,F1
No log,0.696974,0,0.498720,0.333471



DistilBERT Initial Evaluation Results:
{'eval_loss': 0.6969738006591797, 'eval_accuracy': 0.49872, 'eval_f1': 0.33347132795760187}


Epoch,Training Loss,Validation Loss,Accuracy,F1
1,0.043056,0.218847,0.916800,0.916665
2,0.029927,0.222714,0.929080,0.929071
3,0.002444,0.284495,0.931640,0.931631
4,0.003637,0.366731,0.927040,0.926992
5,0.000929,0.357567,0.931720,0.931719


...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Epoch 5 completed in 312.27 seconds


Training Loss,Validation Loss,Epoch,Accuracy,F1
0.000929,0.357567,5,0.931720,0.931719



DistilBERT Evaluation Results:
{'eval_loss': 0.35756716132164, 'eval_accuracy': 0.93172, 'eval_f1': 0.9317193854744693}


In [14]:
fineTuneBert(1024,batch_size=16)

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert/distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
pre_classifier.weight   | MISSING    | 
pre_classifier.bias     | MISSING    | 
classifier.weight       | MISSING    | 
classifier.bias         | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
[transformers] `logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


RuntimeError: The size of tensor a (532) must match the size of tensor b (512) at non-singleton dimension 1

## 4. Approach 2: Mamba

Here, we will implement and train a Mamba model. We can either fine-tune a pre-trained Mamba model (e.g., `state-spaces/mamba-130m` if available for classification) or build a small Mamba model from scratch and train it for sequence classification. This section will require custom model definition and potentially custom training loops if the `Trainer` API isn't directly compatible.

### Fine-tuning Mamba

For Mamba, we will define a custom `MambaForSequenceClassification` model, as `transformers` does not yet have a direct `AutoModelForSequenceClassification` for Mamba. We will initialize a Mamba block and add a classification head on top. The training process will then use the same `Trainer` API.

In [ ]:
from transformers import MambaForSequenceClassification, AutoConfig
import torch.nn as nn
import time
import torch

# Initialize Mamba model for sequence classification
# NOTE: `state-spaces/mamba-130m` was likely pre-trained with a tokenizer like `EleutherAI/gpt-neox-20b`.
# Using `distilbert-base-uncased` tokenizer might lead to sub-optimal results due to vocabulary differences.
# The `transformers` library will attempt to resize the embeddings, but the new embeddings will be random.

mamba_model = MambaForSequenceClassification.from_pretrained(
    "state-spaces/mamba-130m",
    num_labels=2, # For binary classification (positive/negative sentiment)
    # If the tokenizer's vocab_size differs from the pre-trained model's, transformers will resize.
    # We don't explicitly pass vocab_size here as from_pretrained handles it by loading the model's config.
)

# Define training arguments (can be same as Transformer or adjusted)
mamba_training_args = TrainingArguments(
    output_dir="./mamba_results",
    eval_strategy="epoch", # Corrected argument name
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=3,
    weight_decay=0.01,
    logging_dir='./mamba_logs',
    logging_steps=10,
)

# Initialize Trainer for Mamba
mamba_trainer = Trainer(
    model=mamba_model,
    args=mamba_training_args,
    train_dataset=tokenized_dataset["train"],
    eval_dataset=tokenized_dataset["test"],
    # tokenizer=tokenizer, # Removed redundant tokenizer argument
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

# Train the Mamba model and track time
print("\nStarting Mamba model training...")
start_train_time_mamba = time.time()
mamba_train_output = mamba_trainer.train()
end_train_time_mamba = time.time()
training_time_mamba = end_train_time_mamba - start_train_time_mamba

# Evaluate the Mamba model and track time
print("\nStarting Mamba model evaluation...")
start_eval_time_mamba = time.time()
mamba_results = mamba_trainer.evaluate()
end_eval_time_mamba = time.time()
inference_time_mamba = end_eval_time_mamba - start_eval_time_mamba

print("\nMamba Training Statistics:")
print(f"Epoch Training Time: {training_time_mamba / mamba_training_args.num_train_epochs:.2f} seconds per epoch")
print(f"Total Training Time: {training_time_mamba:.2f} seconds")

print("\nMamba Evaluation Results:")
print(f"Inference Time: {inference_time_mamba:.2f} seconds")
print(f"Accuracy: {mamba_results['eval_accuracy']:.4f}")
print(f"F1 Score: {mamba_results['eval_f1']:.4f}")
print(mamba_results)

## 5. Approach 3: Scaling Analysis

This section will focus on comparing both architectures across different sequence lengths (128, 512, 1024 tokens). We will measure:
-   **Training time per epoch**
-   **Inference time**
-   **Quality metrics**: Accuracy, F1-score

This will involve re-tokenizing the dataset with different `max_length` values and repeating the training and evaluation steps for each model and sequence length.

In [ ]:
# Helper function to compute metrics
def compute_metrics(eval_pred):
    load_accuracy = evaluate.load("accuracy")
    load_f1 = evaluate.load("f1")
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    accuracy = load_accuracy.compute(predictions=predictions, references=labels)["accuracy"]
    f1 = load_f1.compute(predictions=predictions, references=labels, average="weighted")["f1"]
    return {"accuracy": accuracy, "f1": f1}